In [16]:
%pip install opencv-python
#Run if you don't have opencv installed.

Note: you may need to restart the kernel to use updated packages.


In [17]:
import cv2
import numpy as np
import os
import time

In [18]:
# paths
INPUT_VIDEO = "input.mp4"
OUTPUT_DIR  = "output_frames"

IMG_HEIGHT = 480
IMG_WIDTH  = 640

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [19]:
# can change based on what color we want
COLOR = [
    (np.array([0, 0, 0]),  np.array([179, 50, 80])),  # lower gray
    (np.array([0, 0, 80]),  np.array([180, 50, 200])),  # upper gray
]

MIN_AREA = 800  # remove small items      
BLUR = (11, 11)  # blur

In [20]:
def mask(hsv: np.ndarray) -> np.ndarray:
    lower = cv2.inRange(hsv, COLOR[0][0], COLOR[0][1])
    upper = cv2.inRange(hsv, COLOR[1][0], COLOR[1][1])
    mask  = cv2.bitwise_or(lower, upper)
 
    # clean up mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

In [21]:
def draw(frame: np.ndarray, mask: np.ndarray) -> np.ndarray:
    output = frame.copy()
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    found = 0

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < MIN_AREA: continue
        found += 1
 
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(output, (x, y), (x + w, y + h), (0, 0, 255), 2)
    return output

## Chop input video into images

In [ ]:
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened(): raise RuntimeError(f"Could not open video: {INPUT_VIDEO}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Video: {total_frames} frames @ {fps:.1f} fps")

frame_idx = 0
times = []

while True:
    ret, frame = cap.read()
    if not ret: break

    # Resize to match HLS block dimensions if needed
    if frame.shape[0] != IMG_HEIGHT or frame.shape[1] != IMG_WIDTH:
        frame = cv2.resize(frame, (IMG_WIDTH, IMG_HEIGHT))

    t0 = time.perf_counter()
    
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 1.4)
    edges = cv2.Canny(blurred, threshold1=50, threshold2=150)

    edge_overlay = frame.copy()
    edge_overlay[edges != 0] = (0, 0, 255)
    # hsv = cv2.cvtColor(cv2.GaussianBlur(frame, BLUR, 0), cv2.COLOR_BGR2HSV)
    # m = mask(hsv)
    # output = draw(frame, m)

    times.append(time.perf_counter() - t0)
    
    out_path = os.path.join(OUTPUT_DIR, f"frame_{frame_idx:05d}.png")
    cv2.imwrite(out_path, edge_overlay)#output)

    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"  Processed {frame_idx}/{total_frames} frames")

cap.release()

avg_ms = (sum(times) / len(times)) * 1000 if times else 0
print(f"\nDone. {frame_idx} frames saved to '{OUTPUT_DIR}/'")
print(f"Avg SW detection time per frame: {avg_ms:.2f} ms")

Video: 91 frames @ 30.0 fps
  Processed 50/91 frames

Done. 91 frames saved to 'output_frames/'
Avg SW detection time per frame: 1.36 ms


## Reconstruct images to video

In [28]:
OUTPUT_VIDEO = "output_edges.avi"

fourcc  = cv2.VideoWriter_fourcc(*'XVID')
writer  = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (IMG_WIDTH, IMG_HEIGHT), isColor=True)

frame_files = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith(".png"))
for fname in frame_files:
    img = cv2.imread(os.path.join(OUTPUT_DIR, fname))
    writer.write(img)

writer.release()
print(f"Saved reassembled video: {OUTPUT_VIDEO}")

Saved reassembled video: output_edges.avi
